# Minimal Coding Agent

Let's build the smallest coding agent that is still useful.

My inspiration for writing this came from Geoffrey Huntley's post https://ghuntley.com/agent/ and the `mini-swe-agent` project at https://github.com/SWE-agent/mini-swe-agent.

The idea is simple:

1. Send the conversation to the model.
2. If the model replies with a fenced `bash` block, run that command.
3. Feed the command output back to the model.
4. Stop when the model replies without a `bash` block.

That is the whole agent.

This notebook is intentionally small and direct. It is for learning, not production.

## What we are using

Only the Python standard library:

- `urllib.request` for the OpenAI HTTP call
- `json` for payloads
- `subprocess` for bash commands
- `re` to find a fenced `bash` block
- `os` for the working directory

We are not using the OpenAI Python SDK here on purpose. The wire format is easier to see this way.

In [ ]:
import json
import os
import re
import subprocess
import urllib.error
import urllib.request


## Step 1: one tiny function to call OpenAI

Use the Responses API over plain HTTP.

We make the API key an explicit variable in the notebook so the user has to paste it locally. Do not commit a real key here.

In [ ]:
api_key = "paste-your-openai-api-key-here"

API_URL = "https://api.openai.com/v1/responses"
MODEL = "gpt-5-mini"


def call_openai(messages, api_key, model=MODEL):
    """Send the current conversation to the OpenAI Responses API."""
    if not api_key or api_key == "paste-your-openai-api-key-here":
        raise RuntimeError("Paste your real OPENAI_API_KEY into the api_key variable first.")

    payload = {
        "model": model,
        "input": messages,
        "store": False,
    }

    request = urllib.request.Request(
        API_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        body = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"OpenAI API error {error.code}: {body}") from error


### Simple test

Before we spend money on an API call, prove that the guard works. This should fail fast because the placeholder key is still there.

In [ ]:
try:
    call_openai([], api_key="")
except RuntimeError as error:
    print(error)


## Step 2: pull the assistant text out of the response

The JSON response is nested. We only want the assistant text.

In [ ]:
def response_text(response_json):
    """Extract plain assistant text from a Responses API JSON object."""
    parts = []

    for item in response_json.get("output", []):
        if item.get("type") != "message":
            continue

        for content in item.get("content", []):
            if content.get("type") == "output_text":
                parts.append(content.get("text", ""))

    return "".join(parts).strip()


### Simple test

Use a fake response object so we can confirm the extraction logic without talking to the network.

In [ ]:
fake_response = {
    "output": [
        {"type": "reasoning", "summary": []},
        {
            "type": "message",
            "content": [
                {"type": "output_text", "text": "hello from the model"}
            ],
        },
    ]
}

print(response_text(fake_response))


## Step 3: define a tiny protocol

We are not doing formal tool calling in this notebook.

Instead, we tell the model:

- If you need to inspect or change the repo, reply with exactly one fenced `bash` block and nothing else.
- If you are done, reply with plain text and no `bash` block.

That gives us a stop condition that is trivial to parse.

In [ ]:
SYSTEM_PROMPT = """You are a tiny coding agent.

When you need to run a shell command, reply with exactly one fenced bash block and nothing else.

Example:
```bash
ls
```

When you are done, reply with plain text and do not include a bash block.
Keep commands small and relevant to the task.
"""

BASH_BLOCK_RE = re.compile(r"```bash\n(.*?)```", re.DOTALL)


def extract_bash_command(text):
    """Return the first fenced bash command, or None if no block exists."""
    match = BASH_BLOCK_RE.search(text)
    if not match:
        return None
    return match.group(1).strip()


### Simple test

Give the parser one assistant message with a bash block and one without it.

In [ ]:
print(extract_bash_command("```bash\npwd\n```"))
print(extract_bash_command("I am done."))


## Step 4: run bash and package the result

We send stdout, stderr, and the exit code back to the model.

Also, we clip long output so the next prompt stays small enough to be practical.

In [ ]:
def clip(text, limit=4000):
    """Trim long command output so we do not stuff huge logs back into the model."""
    if len(text) <= limit:
        return text
    return text[:limit] + "\n...<truncated>"


def run_bash(command, cwd=None, timeout=60):
    """Run one bash command and return its exit code, stdout, and stderr."""
    completed = subprocess.run(
        ["bash", "-lc", command],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    return {
        "returncode": completed.returncode,
        "stdout": clip(completed.stdout),
        "stderr": clip(completed.stderr),
    }


def format_bash_result(command, result):
    """Format one shell execution result as the next user message."""
    return f"""Command:
{command}

Exit code: {result['returncode']}

Stdout:
{result['stdout']}

Stderr:
{result['stderr']}
"""


### Simple test

Run a tiny shell command and show the formatted result. This proves the shell bridge works before the model gets involved.

In [ ]:
shell_result = run_bash("printf 'hello from bash'", timeout=60)
print(shell_result)
print()
print(format_bash_result("printf 'hello from bash'", shell_result))


## Step 5: the agent loop

This is the whole agent.

We keep a normal conversation history. If the assistant gives us a bash block, we execute it and feed the result back as the next user message.

In [ ]:
def run_agent(task, api_key, cwd=None, max_steps=8, timeout=60):
    """Run the minimal agent loop until the model stops asking for bash."""
    cwd = cwd or os.getcwd()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Working directory: {cwd}\n\nTask: {task}",
        },
    ]

    for step in range(1, max_steps + 1):
        response_json = call_openai(messages, api_key=api_key)
        assistant_message = response_text(response_json)

        print(f"\nStep {step} assistant:\n")
        print(assistant_message)

        messages.append({"role": "assistant", "content": assistant_message})

        command = extract_bash_command(assistant_message)
        if command is None:
            print("\nNo bash block found. Stopping.")
            return assistant_message

        print(f"\n$ {command}\n")
        result = run_bash(command, cwd=cwd, timeout=timeout)

        if result["stdout"]:
            print(result["stdout"])
        if result["stderr"]:
            print(result["stderr"])

        messages.append(
            {"role": "user", "content": format_bash_result(command, result)}
        )

    raise RuntimeError("Agent hit max_steps before finishing.")


### Simple test

Now run the real thing.

Paste your real key into `api_key` first. Then run this cell. The task is intentionally concrete: write a C++ hello-world program, compile it, test it, and run it.

In [ ]:
task = "Write a C++ program that prints `Hello World`. Make sure you compile it, test it and run it."

# run_agent(task, api_key=api_key)


## Inspect what the agent created

After the agent runs, list the current directory. This makes the created files visible.

In [ ]:
!ls


## Run the program yourself

The final check is manual. Run the compiled program yourself and see `Hello World` come back.

If your agent chose a different binary name, adjust this command.

In [ ]:
!./hello


## Optional example: Lorenz attractor

This example is a little less minimal because plotting needs `matplotlib`.

That is fine here. The agent loop is still the same. Only the task changes.

In this repo, `matplotlib` is already part of the environment. Now ask the agent to create a Python script that solves the Lorenz equations and saves a plot to `lorenz.png`.

In [ ]:
task = "Write a Python script that numerically solves the Lorenz equations and saves a plot of the attractor to `lorenz.png`. Make sure you run the script and confirm that the image file was created."

# run_agent(task, api_key=api_key)


## Show the Lorenz plot

After the agent runs, load the saved image and display it in the notebook.

If the agent chose a different filename, adjust the path.

In [ ]:
from IPython.display import Image, display

display(Image(filename="lorenz.png"))


## Final example: use the agent to build another agent

For a bigger task, ask the Python notebook agent to write a tiny CLI-based coding agent in bash.

Keep it practical: ask for a single script named `bash_agent.sh`, make it executable, and make it read `OPENAI_API_KEY` from the environment.

Before you build and test the bash agent, set `OPENAI_API_KEY` in the notebook environment. Paste your real key here locally, but do not save a real key into the notebook file.

In [ ]:
%env OPENAI_API_KEY=paste-your-openai-api-key-here


In [ ]:
task = """Write a tiny CLI-based coding agent in bash.

Requirements:
- Put the code in `bash_agent.sh`.
- Make it executable.
- Read the user task from the command line.
- Read `OPENAI_API_KEY` from the environment.
- The program should ask OpenAI for a response, run a bash command if the response contains one fenced `bash` block, feed the result back, and stop when there is no bash block.
- Set `max_steps` to 50.
- Keep it small.
- Use `curl` for the HTTP call.
- Use `jq` for JSON parsing.
- After writing the file, run one tiny smoke test.
"""

# run_agent(task, api_key=api_key)


## Test the generated bash agent

Now use the generated bash agent on a new task: create CSV data for the bifurcation diagram of the logistic map.

If the agent chose different filenames, adjust the command.

In [ ]:
bash_task = "Create a CSV file named `logistic_bifurcation.csv` with columns `r,x` containing data for the bifurcation diagram of the logistic map. Use enough samples to make the structure visible. Do not make a plot."

subprocess.run(
    ["bash", "./bash_agent.sh", bash_task],
    check=True,
    text=True,
)


## Plot the generated data

If the bash agent did its job, you should now have `logistic_bifurcation.csv`. Load it and draw the bifurcation diagram.

In [ ]:
import csv

import matplotlib.pyplot as plt

r_values = []
x_values = []

with open("logistic_bifurcation.csv", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        r_values.append(float(row["r"]))
        x_values.append(float(row["x"]))

plt.figure(figsize=(8, 6))
plt.scatter(r_values, x_values, s=0.05, color="black")
plt.xlabel("r")
plt.ylabel("x")
plt.title("Logistic Map Bifurcation Diagram")
plt.show()


## Official docs behind the wire format

- API auth overview: https://developers.openai.com/api/reference/overview
- Responses is recommended for new projects: https://developers.openai.com/api/docs/guides/migrate-to-responses
- Current model list: https://platform.openai.com/docs/models
